# Source code - Public Perception of Health Care Before, During, and After COVID-19

This notebook contains the source code used in the longitudinal analysis of citizen-generated online reviews of the Andalusian Public Health System (2016–2025), as reported in the manuscript submitted to the *Journal of Medical Internet Research*.

## Pipeline overview

This notebook implements the analytical procedures supporting the results reported in the manuscript. The workflow is organized as follows:

0. **Environment setup** — import of the Python libraries required for data processing, statistical analysis, change-point detection, and visualization.

1. **Data loading and initial preprocessing** — loading of the raw review-level dataset, variable normalization, date processing, and creation of the star-rating-based sentiment classification.

3. **Validation of star-rating-based sentiment classification** — manual annotation of a stratified sample of 300 reviews and estimation of agreement between manual labels and star-rating-derived sentiment categories using linearly weighted Cohen's kappa.

5. **Construction of the analytical textual corpus** — exclusion of reviews without sufficient textual content and application of the preprocessing steps required for the analytical dataset.


4. **Structural exploratory data analysis** — descriptive examination of the raw review dataset, including the annual volume of retrieved reviews before application of the final analytical inclusion criteria.

6. **Change-point detection** — identification of structural breaks in the monthly proportion of negative reviews using the Pruned Exact Linear Time (PELT) algorithm with a radial basis function (RBF) cost function.

7. **Global logistic regression models** — estimation of nested logistic regression models assessing associations between level of care, temporal trend, and the probability of a negative review, with standard errors clustered at the facility level.

8. **Pandemic-period analysis** — comparison of pre-pandemic, pandemic, and post-pandemic phases; estimation of the interaction between level of care and pandemic period; visualization of temporal trajectories by level of care; and likelihood ratio testing of the interaction model.

## Data availability

The underlying review-level dataset is not redistributed in this repository because it contains user-generated review text retrieved from Google Maps. Public redistribution of the raw scraped corpus is restricted in accordance with the Google Maps Platform Terms of Service, and verbatim review-level content may also raise privacy considerations.

The notebook reads from a local input file, `Merged.csv`, which is not included in the public repository. The manual annotation file used for validation is also not publicly distributed because it contains review-level textual content.

The manuscript reports the aggregated results derived from the analytical dataset, while this repository provides the source code used for data preprocessing, sentiment classification validation, change-point detection, statistical modeling, and figure generation.

## License

The source code in this repository is released under the MIT License. See the `LICENSE` file for details.

## Citation

If you use this code, please cite the corresponding manuscript:

Gazquez-Garcia J, Sánchez-Bocanegra CL, Fernández-Llatas C. *Public Perception of Health Care Before, During, and After COVID-19: A Longitudinal Analysis of Online Reviews*. Manuscript submitted for publication.


# 00_Environment

In [ ]:
# Dependencies
!pip install -q nltk ruptures openpyxl
!python -m spacy download es_core_news_sm

# General
import numpy as np
import pandas as pd
import re
import os

# NLP
import nltk
from nltk.corpus import stopwords
import spacy

# Statistical analysis and models
from scipy.stats import chi2
import statsmodels.formula.api as smf
from sklearn.metrics import (cohen_kappa_score, classification_report, confusion_matrix, ConfusionMatrixDisplay)

# Change-point detection
import ruptures as rpt

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# NLTK resources
nltk.download("stopwords", quiet=True)
nltk.download("punkt", quiet=True)

# Spanish language model
nlp = spacy.load("es_core_news_sm")

# Display options
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 100)

print("Environment ready")

# 01_Raw data loading and initial preprocessing

In [ ]:
data = pd.read_csv("Merged.csv", sep=";")
print(f"Filas cargadas: {len(data):,}")
data.head()
data.info()

In [ ]:
data["stars"] = pd.to_numeric(data["stars"], errors="coerce")
data = data[data["stars"].between(1, 5)]

data["Fecha"] = pd.to_datetime(data["Fecha"], errors="coerce")
data["year"]  = data["Fecha"].dt.year

In [ ]:
for col in ["Atencion", "Ciudad", "Municipio",
            "Tipo_de_centro", "Dependencia", "Nombre"]:
    if col in data.columns:
        data[col] = data[col].astype("string").str.strip()

data["Tipo_de_centro"]      = data["Tipo_de_centro"].str.lower()
data["Tipo_de_centro_norm"] = data["Tipo_de_centro"].copy()
data["Atencion"]            = data["Atencion"].replace(
                                "Especializada", "Hospitalaria")

print(f"Filas tras limpieza: {len(data):,}")
data.info()

In [ ]:
def etiqueta_heuristica(stars):
    if stars <= 2:   return "negativo"
    elif stars == 3: return "neutro"
    else:            return "positivo"

data["sentimiento_heur"] = data["stars"].apply(etiqueta_heuristica)

# 02_Validation of star-rating-based sentiment categorization
A stratified sample of 300 reviews was manually annotated to assess agreement with the star-rating-based sentiment classification.

In [ ]:
data_con_texto = data[
    data["text"].notna() &
    (data["text"].str.strip() != "")
].copy()

print(f"Reseñas con texto: {len(data_con_texto):,} (de {len(data):,} totales)")

np.random.seed(42)
muestra = (
    data_con_texto
    .groupby("sentimiento_heur", group_keys=False)
    .apply(lambda x: x.sample(min(100, len(x)), random_state=42))
    [["text", "stars", "totalScore", "sentimiento_heur", "Atencion"]]
    .reset_index(drop=True)
)
muestra["etiqueta_manual"] = ""
muestra.to_excel("validacion_manual.xlsx", index=True)
print(f"Exportadas {len(muestra)} reseñas → validacion_manual.xlsx")

In [ ]:
val = pd.read_excel("validacion_manual.xlsx", index_col=0)

mapeo = {1.0: "negativo", 2.0: "negativo",
         3.0: "neutro",
         4.0: "positivo", 5.0: "positivo"}
val["etiqueta_manual_txt"] = val["etiqueta_manual"].map(mapeo)
val = val[val["etiqueta_manual_txt"].notna()]

print(f"Reseñas anotadas válidas: {len(val)}")
print(val["etiqueta_manual_txt"].value_counts())

kappa = cohen_kappa_score(
    val["sentimiento_heur"],
    val["etiqueta_manual_txt"],
    weights="linear"
)

if kappa > 0.80:   nivel = "Acuerdo casi perfecto"
elif kappa > 0.60: nivel = "Acuerdo sustancial"
elif kappa > 0.40: nivel = "Acuerdo moderado"
else:              nivel = "Acuerdo débil — revisar criterios"

print(f"\nKappa de Cohen (lineal): {kappa:.3f} — {nivel}")
print(classification_report(
    val["sentimiento_heur"],
    val["etiqueta_manual_txt"],
    labels=["negativo", "neutro", "positivo"]
))

In [ ]:
cm = confusion_matrix(
    val["sentimiento_heur"],
    val["etiqueta_manual_txt"],
    labels=["negativo", "neutro", "positivo"]
)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["negativo", "neutro", "positivo"],
            yticklabels=["negativo", "neutro", "positivo"], ax=ax)
ax.set_xlabel("Etiqueta manual", fontsize=12)
ax.set_ylabel("Etiqueta heurística", fontsize=12)
ax.set_title(f"Matriz de confusión · Kappa = {kappa:.3f} ({nivel})",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig("validacion_kappa.png", dpi=300, bbox_inches="tight")
plt.savefig("validacion_kappa.svg")
plt.show()


#### Validation of Star-Rating-Based Sentiment Classification

The star-rating-based sentiment classification was validated through manual annotation of a stratified sample of 300 reviews, with 100 reviews selected from each sentiment category. Agreement between the automatic classification based on star ratings and the manual annotation reached a linearly weighted Cohen's kappa of 0.84, indicating high agreement. Overall accuracy was 86%.

# 03_Construction of the analytical textual corpus

In [ ]:
data["length_words"] = (
    data["text"].fillna("").apply(lambda x: len(str(x).split()))
)

corpus = data[data["length_words"] >= 5].copy()
print(f"Corpus NLP: {len(corpus):,} reseñas (≥5 palabras)")

corpus["text"] = corpus["text"].astype("string")

In [ ]:
def limpiar_espanol(text):
    text = str(text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#\w+', '', text)
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r"[^a-zA-ZáéíóúÁÉÍÓÚñÑüÜ\s]", " ", text)
    text = re.sub(r'\s+', ' ', text).strip().lower()
    return text

corpus["text_clean"] = corpus["text"].apply(limpiar_espanol)

stop_es = set(stopwords.words("spanish"))

def tokens_filtrados(text):
    return [t for t in text.split()
            if t not in stop_es and len(t) > 2]

stop_en = {"the", "and", "was", "were", "is", "are",
           "to", "of", "in", "on"}

def es_ingles(text):
    toks = text.split()
    if not toks:
        return False
    return sum(1 for t in toks if t in stop_en) / len(toks) > 0.2

corpus = corpus[~corpus["text_clean"].apply(es_ingles)].copy()

print(f"Corpus final tras filtro inglés: {len(corpus):,} reseñas")
corpus[["text", "text_clean"]].sample(5, random_state=42)

In [ ]:
MIN_FREQ_RATIO = 100
TOP_N          = 15
TOP_N_GLOBAL   = 30

def clasificar_sentimiento(p):
    if pd.isna(p):
        return np.nan
    if p >= 4:
        return "positivo"
    elif p <= 2:
        return "negativo"
    else:
        return "neutro"

corpus["sentimiento"] = corpus["stars"].apply(clasificar_sentimiento)

n_neg   = len(corpus[corpus["sentimiento"] == "negativo"])
n_pos   = len(corpus[corpus["sentimiento"] == "positivo"])
n_total = len(corpus)

print(f"Distribución de sentimiento (corpus NLP):")
print(f"  Positivo : {n_pos:>6,}  ({n_pos/n_total*100:.1f}%)")
print(f"  Neutro   : {n_total - n_neg - n_pos:>6,}  ({(n_total - n_neg - n_pos)/n_total*100:.1f}%)")
print(f"  Negativo : {n_neg:>6,}  ({n_neg/n_total*100:.1f}%)")
print(f"  TOTAL    : {n_total:>6,}")

# 04_Structural Exploratory Data Analysis (EDA)


In [ ]:
freq_stars = data["stars"].value_counts().sort_index()

plt.figure(figsize=(7, 4))
plt.bar(freq_stars.index.astype(int), freq_stars.values)
plt.title("Distribución de valoraciones")
plt.xlabel("Estrellas")
plt.ylabel("Frecuencia")
plt.grid(alpha=0.2)
plt.tight_layout()
plt.savefig("eda_stars.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
n_centros = corpus["Nombre"].nunique()
print(f"Número de centros únicos: {n_centros}")

In [ ]:
porcentajes = corpus["Atencion"].value_counts(normalize=True) * 100

print(porcentajes.round(2))

In [ ]:
data_plot = data[(data["year"] >= 2016) & (data["year"] < 2026)]

vol_year = data_plot["year"].value_counts().sort_index()

plt.figure(figsize=(9,4))
plt.plot(vol_year.index, vol_year.values, marker="o", linewidth=2)

plt.title("Annual Number of Reviews")
plt.xlabel("Year")
plt.ylabel("Number of Reviews")

plt.xticks(vol_year.index)
plt.grid(alpha=0.2)

plt.tight_layout()
plt.savefig("annual_number_reviews.png", dpi=300, bbox_inches="tight")
plt.show()

## 05_Change-Point Detection (PELT + RBF) 

Applied to monthly time series of negative review proportions

In [ ]:

if "negativa" not in corpus.columns:
    corpus["negativa"] = (corpus["sentimiento"] == "negativo").astype(int)

if "year" not in corpus.columns:
    corpus["year"] = pd.to_datetime(corpus["Fecha"], errors="coerce").dt.year
    corpus = corpus.dropna(subset=["year"])
    corpus["year"] = corpus["year"].astype(int)

if "year_c" not in corpus.columns:
    corpus["year_c"] = corpus["year"] - corpus["year"].mean()

corpus["Atencion"] = corpus["Atencion"].astype(str).str.strip()

print("Variables ready")
print(f"  Negative reviews: {corpus['negativa'].mean()*100:.1f}%")
print(f"  Year range: {corpus['year'].min()} – {corpus['year'].max()}")



In [ ]:
# ─────────────────────────────────────────
# TIME SERIES PREPARATION
# ─────────────────────────────────────────
df_ts = corpus.copy()
df_ts["Atencion"] = df_ts["Atencion"].astype(str).str.strip()
df_ts = df_ts[df_ts["Atencion"].isin(["Primaria", "Hospitalaria"])]

df_ts["year_month"] = pd.to_datetime(df_ts["Fecha"], errors="coerce").dt.to_period("M")
df_ts = df_ts.dropna(subset=["year_month"])

df_ts = df_ts[(df_ts["year"] >= 2016) & (df_ts["year"] <= 2025)].copy()

ts = (
    df_ts.groupby(["year_month", "Atencion"])
    .agg(n_neg=("negativa", "sum"), n_total=("negativa", "count"))
    .reset_index()
)

ts["pct_neg"] = (ts["n_neg"] / ts["n_total"] * 100).round(2)
ts = ts.sort_values("year_month").reset_index(drop=True)

print(f"Time series: {len(ts)} observations (month × care level)")
print(f"Range: {ts['year_month'].min()} → {ts['year_month'].max()}")



In [ ]:
# ─────────────────────────────────────────
# CHANGE-POINT DETECTION (PELT + RBF)
# ─────────────────────────────────────────
resultados_cp = {}

for nivel in ["Primaria", "Hospitalaria"]:
    serie  = ts[ts["Atencion"] == nivel]["pct_neg"].values
    fechas = ts[ts["Atencion"] == nivel]["year_month"].values

    algo = rpt.Pelt(model="rbf").fit(serie)
    breakpoints = algo.predict(pen=3)

    cp_indices = [b for b in breakpoints if b < len(serie)]
    cp_fechas  = [str(fechas[i]) for i in cp_indices]

    resultados_cp[nivel] = {
        "indices": cp_indices,
        "fechas": cp_fechas,
        "serie": serie,
        "fechas_completas": fechas
    }

    print(f"\n{nivel}: {len(cp_indices)} change point(s) detected")
    for idx, fecha in zip(cp_indices, cp_fechas):
        print(f"  → {fecha} (index {idx})")

In [ ]:

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

colors = {"Primaria": "#C0392B", "Hospitalaria": "#2980B9"}
titles = {"Primaria": "Primary care", "Hospitalaria": "Hospital care"}

for ax, nivel in zip(axes, ["Primaria", "Hospitalaria"]):
    sub = ts[ts["Atencion"] == nivel].copy()
    x = np.arange(len(sub))
    y = sub["pct_neg"].values
    labels_x = sub["year_month"].astype(str).values

  
    ax.plot(x, y, color=colors[nivel], linewidth=2.5, zorder=3)
    ax.fill_between(x, y, alpha=0.12, color=colors[nivel], zorder=2)


    covid_start_idx = next((i for i, f in enumerate(labels_x) if f == "2020-03"), None)
    covid_end_idx   = next((i for i, f in enumerate(labels_x) if f == "2022-05"), None)

    if covid_start_idx is not None and covid_end_idx is not None:
        ax.axvspan(covid_start_idx, covid_end_idx, color="grey", alpha=0.08, zorder=1)

    # Detected change points
    for idx in resultados_cp[nivel]["indices"]:
        ax.axvline(
            x=idx,
            color="black",
            linestyle="--",
            linewidth=1.8,
            alpha=0.75,
            zorder=4
        )
        ax.text(
            idx + 1,
            ax.get_ylim()[1] * 0.90,
            labels_x[idx],
            fontsize=9,
            color="black",
            rotation=45,
            ha="left",
            va="bottom",
            fontweight="bold"
        )

    year_ticks = [i for i, f in enumerate(labels_x) if f.endswith("-01")]
    ax.set_xticks(year_ticks)
    ax.set_xticklabels([labels_x[i][:4] for i in year_ticks], fontsize=10)

    ax.set_ylabel("Negative reviews (%)", fontsize=11)
    ax.set_title(titles[nivel], fontsize=13, fontweight="bold", pad=8)

    ax.grid(axis="y", alpha=0.25, linestyle=":")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)


axes[-1].set_xlabel("Year", fontsize=11)

plt.suptitle(
    "Structural change-point detection in negative review trajectories",
    fontsize=15,
    fontweight="bold",
    y=0.98
)

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig("changepoint_temporal_en_clean.png", dpi=300, bbox_inches="tight")
plt.savefig("changepoint_temporal_en_clean.svg", bbox_inches="tight")
plt.show()

print("Saved: changepoint_temporal_en_clean.png and .svg")

# 06_Global logistic regression models
#### Factors associated with the probability of a negative review

In [ ]:

vars_necesarias = ["negativa", "year", "year_c", "Atencion"]
vars_faltantes  = [v for v in vars_necesarias if v not in corpus.columns]

if vars_faltantes:
    raise ValueError(f"Variables no encontradas en corpus: {vars_faltantes}\n"
                     f"   Ejecuta el bloque 8 antes de continuar.")

print(f"Variables verificadas")
print(f"   negativa : {corpus['negativa'].mean()*100:.1f}% negativas")
print(f"   year     : {corpus['year'].min()} – {corpus['year'].max()}")
print(f"   year_c   : centrado en {corpus['year'].mean():.1f}")
print(f"   n total  : {len(corpus):,}")

In [ ]:
df_m1 = corpus[["negativa", "Atencion", "Nombre"]].dropna()

modelo1_v2 = smf.logit(
    "negativa ~ C(Atencion)",
    data=df_m1
).fit(
    cov_type="cluster",
    cov_kwds={"groups": df_m1["Nombre"]}
)

print("\n" + "="*60)
print("MODELO 1 — Solo nivel asistencial (clustered SE)")
print("="*60)
print(modelo1_v2.summary())

or1 = pd.DataFrame({
    "Variable": modelo1_v2.params.index,
    "OR":       np.exp(modelo1_v2.params),
    "CI_low":   np.exp(modelo1_v2.conf_int()[0]),
    "CI_high":  np.exp(modelo1_v2.conf_int()[1]),
    "p":        modelo1_v2.pvalues
}).round(3)

print("\nOdds Ratios — modelo1_v2:")
display(or1)

or1.to_csv("modelo1_v2_odds_ratios.csv", index=False)

In [ ]:
df_m2 = corpus[["negativa", "Atencion", "year_c", "Nombre"]].dropna()

modelo2_v2 = smf.logit(
    "negativa ~ C(Atencion) + year_c",
    data=df_m2
).fit(
    cov_type="cluster",
    cov_kwds={"groups": df_m2["Nombre"]}
)

print("\n" + "="*60)
print("MODELO 2 — Ajustado por año (centrado, clustered SE)")
print("="*60)
print(modelo2_v2.summary())

or2 = pd.DataFrame({
    "Variable": modelo2_v2.params.index,
    "OR":       np.exp(modelo2_v2.params),
    "CI_low":   np.exp(modelo2_v2.conf_int()[0]),
    "CI_high":  np.exp(modelo2_v2.conf_int()[1]),
    "p":        modelo2_v2.pvalues
}).round(3)

print("\nOdds Ratios — Modelo 2:")
display(or2)

or2.to_csv("modelo2_odds_ratios.csv", index=False)

In [ ]:
df_m3 = corpus[["negativa", "Atencion", "year_c", "Nombre"]].dropna()

modelo3_v2 = smf.logit(
    "negativa ~ C(Atencion) * year_c",
    data=df_m3
).fit(
    cov_type="cluster",
    cov_kwds={"groups": df_m3["Nombre"]}
)

print("\n" + "="*60)
print("MODELO 3 — Interacción Atención × Tiempo (clustered SE)")
print("="*60)
print(modelo3_v2.summary())

or3 = pd.DataFrame({
    "Variable": modelo3_v2.params.index,
    "OR":       np.exp(modelo3_v2.params),
    "CI_low":   np.exp(modelo3_v2.conf_int()[0]),
    "CI_high":  np.exp(modelo3_v2.conf_int()[1]),
    "p":        modelo3_v2.pvalues
}).round(3)

print("\nOdds Ratios — Modelo 3:")
display(or3)

or3.to_csv("modelo3_odds_ratios.csv", index=False)

In [ ]:
# ─────────────────────────────────────────
# 9.4 COMPARACIÓN DE MODELOS
# ─────────────────────────────────────────
comparacionv2 = pd.DataFrame({
    "Modelo":    ["M1: Solo Atención", "M2: Atención + Año", "M3: Interacción"],
    "n":         [len(df_m1), len(df_m2), len(df_m3)],
    "Log-Lik":   [modelo1_v2.llf,       modelo2_v2.llf,       modelo3_v2.llf],
    "AIC":       [modelo1_v2.aic,       modelo2_v2.aic,       modelo3_v2.aic],
    "BIC":       [modelo1_v2.bic,       modelo2_v2.bic,       modelo3_v2.bic],
    "Pseudo-R²": [modelo1_v2.prsquared, modelo2_v2.prsquared, modelo3_v2.prsquared]
}).round(2)

print("\nComparación de modelos:")
display(comparacionv2)

In [ ]:
# ─────────────────────────────────────────
# TABLE — Logistic regression models clustered
# ─────────────────────────────────────────

def formatear_modelo(modelo):
    
    OR  = np.exp(modelo.params)
    CI  = np.exp(modelo.conf_int())
    p   = modelo.pvalues
    
    tabla = pd.DataFrame({
        "OR": OR.round(3),
        "CI_low": CI[0].round(3),
        "CI_high": CI[1].round(3),
        "p": p
    })
    
    tabla["OR (95% CI)"] = tabla["OR"].astype(str) + " (" + tabla["CI_low"].astype(str) + "–" + tabla["CI_high"].astype(str) + ")"
    
    return tabla[["OR (95% CI)", "p"]]


# Extraer resultados
m1 = formatear_modelo(modelo1_v2)
m2 = formatear_modelo(modelo2_v2)
m3 = formatear_modelo(modelo3_v2)


# Construir tabla final
tabla_modelosv2 = pd.concat([
    m1["OR (95% CI)"],
    m2["OR (95% CI)"],
    m3["OR (95% CI)"]
], axis=1)

tabla_modelosv2.columns = [
    "Model 1",
    "Model 2",
    "Model 3"
]


# Renombrar variables
tabla_modelosv2.index = tabla_modelosv2.index.map({
    "C(Atencion)[T.Primaria]": "Primary care",
    "year_c": "Year (centered)",
    "C(Atencion)[T.Primaria]:year_c": "Primary care × Year",
    "Intercept": "Intercept"
})


# Eliminar intercept si no quieres mostrarlo
tabla_modelosv2 = tabla_modelosv2.drop("Intercept")


tabla_modelosv2.index.name = "Variable"

display(tabla_modelosv2)


# Exportar
tabla_modelosv2.to_excel("table_logistic_models.xlsx")

print("Saved: table_logistic_models.xlsx")

### Model comparison

Nested logistic regression models were compared to assess whether the inclusion of temporal trend and the interaction between level of care and time improved model fit. Model comparison was based on the Akaike Information Criterion (AIC), Bayesian Information Criterion (BIC), log-likelihood, pseudo-R², and likelihood ratio testing where appropriate.

The interaction model was used to examine whether temporal changes in the probability of a negative review differed between primary care and hospital services.

# 07_Pandemic-period comparison

In [ ]:
def clasificar_periodo(year):
    
    if year <= 2019:
        return "pre"
    
    elif 2020 <= year <= 2022:
        return "covid"
    
    else:
        return "post"

corpus["periodo_covid"] = corpus["year"].apply(clasificar_periodo)

In [ ]:
corpus["periodo_covid"] = pd.Categorical(
    corpus["periodo_covid"],
    categories=["pre", "covid", "post"],
    ordered=True
)
pd.crosstab(corpus["periodo_covid"], corpus["Atencion"])


In [ ]:
corpus.groupby("periodo_covid")["negativa"].mean()

### 07.1_Temporal evolution and descriptive period table

In [ ]:
# Proporción de negativas por año
neg_year = corpus.groupby("year")["negativa"].mean()

# eliminar 2026 si existe
neg_year = neg_year.loc[2016:2025]

plt.figure(figsize=(9,4))
plt.plot(neg_year.index, neg_year.values * 100, marker="o", linewidth=2)

plt.title("Temporal evolution of the proportion of negative reviews")
plt.xlabel("Year")
plt.ylabel("Negative Reviews (%)")

plt.xticks(neg_year.index)
plt.ylim(30,80)
plt.grid(alpha=0.2)

plt.axvline(2020, linestyle="--", color="red", alpha=0.6)
plt.axvline(2023, linestyle="--", color="red", alpha=0.6)

plt.tight_layout()
plt.savefig("temporal_evolution_negative_reviews2.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
summary_periods = (
    corpus.groupby(["periodo_covid", "Atencion"])
    .agg(
        pct_negative=("negativa", "mean"),
        n_reviews=("negativa", "count")
    )
    .reset_index()
)

summary_periods["pct_negative"] = summary_periods["pct_negative"] * 100

summary_periods

In [ ]:
table_periods = (
    summary_periods
    .pivot(index="periodo_covid", columns="Atencion", values="pct_negative")
    .round(1)
)

table_periods

In [ ]:
# ─────────────────────────────────────────
# TABLE — Negative reviews by pandemic period
# ─────────────────────────────────────────

table_periods_en = (
    summary_periods
    .pivot(index="periodo_covid", columns="Atencion", values="pct_negative")
    .round(1)
)

table_periods_en.index = table_periods_en.index.map({
    "pre": "Pre-pandemic",
    "covid": "Pandemic",
    "post": "Post-pandemic"
})

table_periods_en = table_periods_en.rename(columns={
    "Primaria": "Primary care",
    "Hospitalaria": "Hospital care"
})

table_periods_en = table_periods_en[["Primary care", "Hospital care"]]

table_periods_en.index.name = "Period"

table_periods_en.columns.name = None

print("Table — Proportion of negative reviews by pandemic period")
display(table_periods_en)

table_periods_en.to_excel("table_negative_reviews_periods.xlsx")

print("Saved: table_negative_reviews_periods.xlsx")

### 07.2_Period-based interaction model with cluster-robust standard errors

In [ ]:

df_covid = corpus[["negativa", "Atencion", "periodo_covid", "Nombre"]].dropna()

modelo_covid = smf.logit(
    "negativa ~ C(Atencion) * C(periodo_covid)",
    data=df_covid
).fit(
    cov_type="cluster",
    cov_kwds={"groups": df_covid["Nombre"]}
)

print(modelo_covid.summary())

In [ ]:
tabla_covid = pd.DataFrame({
    "Variable": modelo_covid.params.index,
    "OR": np.exp(modelo_covid.params),
    "CI_low": np.exp(modelo_covid.conf_int()[0]),
    "CI_high": np.exp(modelo_covid.conf_int()[1]),
    "p": modelo_covid.pvalues
})

tabla_covid = tabla_covid[tabla_covid["Variable"] != "Intercept"].copy()

tabla_covid["Variable"] = tabla_covid["Variable"].replace({
    "C(Atencion)[T.Primaria]": "Primary care",
    "C(periodo_covid)[T.covid]": "Pandemic period",
    "C(periodo_covid)[T.post]": "Post-pandemic period",
    "C(Atencion)[T.Primaria]:C(periodo_covid)[T.covid]": "Primary care × Pandemic",
    "C(Atencion)[T.Primaria]:C(periodo_covid)[T.post]": "Primary care × Post-pandemic"
})

tabla_covid["OR"] = tabla_covid["OR"].round(2)
tabla_covid["CI_low"] = tabla_covid["CI_low"].round(2)
tabla_covid["CI_high"] = tabla_covid["CI_high"].round(2)

tabla_covid["OR (95% CI)"] = (
    tabla_covid["OR"].astype(str)
    + " ("
    + tabla_covid["CI_low"].astype(str)
    + "–"
    + tabla_covid["CI_high"].astype(str)
    + ")"
)

tabla_covid["p-value"] = tabla_covid["p"].apply(
    lambda x: "<0.001" if x < 0.001 else f"{x:.3f}"
)

tabla_covid_final = tabla_covid[["Variable", "OR (95% CI)", "p-value"]]

display(tabla_covid_final)
tabla_covid_final.to_excel("table_covid_model_clustered.xlsx", index=False)

#### Interpretation of the Period-Based Interaction Model

The period-based interaction model was used to assess whether changes in the odds of negative reviews differed between primary care and hospital services across the pre-pandemic, pandemic, and post-pandemic phases.

The interaction terms indicated that primary care experienced a substantially greater increase in negative reviews during the pandemic period than hospital care, and that this differential pattern persisted in the post-pandemic period. Odds ratios and cluster-robust 95% confidence intervals are generated in the preceding analytical cells.

### 07.3_Temporal trajectories by level of care


In [ ]:
df_plot = (
    corpus
    .groupby(["year", "Atencion"])
    .agg(
        pct_negative=("negativa", "mean"),
        n=("negativa", "size")
    )
    .reset_index()
)

df_plot["pct_negative"] = df_plot["pct_negative"] * 100

df_plot["Atencion"] = df_plot["Atencion"].replace({
    "Primaria": "Primary care",
    "Hospitalaria": "Hospital care"
})

df_plot = df_plot[df_plot["year"] >= 2016]

sns.set_style("whitegrid")

plt.figure(figsize=(9,5))

sns.lineplot(
    data=df_plot,
    x="year",
    y="pct_negative",
    hue="Atencion",
    marker="o",
    linewidth=2
)

plt.axvline(
    x=2020,
    color="black",
    linestyle="--",
    alpha=0.7,
    label="COVID onset"
)

plt.ylabel("Negative reviews (%)")
plt.xlabel("Year")

plt.ylim(0,100)
plt.xlim(2016,2025)

plt.legend(title="Level of care")

plt.tight_layout()
plt.savefig("temporal_trajectories_by_level_of_care.png", dpi=300, bbox_inches="tight")

plt.show()

### 07.4_Likelihood Ratio Test for the Period-Based Interaction Model


In [ ]:
modelo_base = smf.logit(
    "negativa ~ C(Atencion)",
    data=corpus
).fit()

modelo_interaccion = smf.logit(
    "negativa ~ C(Atencion) * C(periodo_covid)",
    data=corpus
).fit()

In [ ]:

LR = 2 * (modelo_interaccion.llf - modelo_base.llf)

df = modelo_interaccion.df_model - modelo_base.df_model

p_value = chi2.sf(LR, df)

print("LR statistic:", LR)
print("df:", df)
print("p-value:", p_value)

To assess whether the full period-based model provided a better fit than a model including level of care alone, a likelihood ratio test was conducted comparing the baseline model with the model including pandemic period and its interaction with level of care.

The full period-based model significantly improved model fit (LR = 3144.83, df = 4, p < 0.001), indicating that temporal period and its interaction with level of care contributed significantly to explaining variation in the probability of a negative review.